## PENDEKATAN 3 VADER TRANSLATION BASED (VTB)

In [1]:
import sys
!{sys.executable} -m pip install deep-translator tqdm
!{sys.executable} -m pip install ipywidgets


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
import time
import warnings
from tqdm import tqdm  
from deep_translator import GoogleTranslator

warnings.filterwarnings('ignore')

# Konfigurasi path
DATA_PATH = '../../outputs/evaluation/ground_truth_final.csv'
OUTPUT_DIR = '../../outputs/VTB'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [3]:
# 2.2 Load Data Ground Truth
df = pd.read_csv(DATA_PATH)

# Validasi kolom
required_cols = ['no', 'timestamp', 'teks', 'teks_processed', 'ground_truth_mean', 'ground_truth_scaled', 'sd_annotator', 'ground_truth_category']
assert all(col in df.columns for col in required_cols), f"Kolom wajib {required_cols} tidak ditemukan!"

print(f"\nData berhasil dimuat: {len(df)} tweet")
print(f"Kolom tersedia: {df.columns.tolist()}")
df.head()


Data berhasil dimuat: 450 tweet
Kolom tersedia: ['no', 'timestamp', 'teks', 'teks_processed', 'ground_truth_mean', 'ground_truth_scaled', 'sd_annotator', 'ground_truth_category']


,no,timestamp,teks,teks_processed,ground_truth_mean,ground_truth_scaled,sd_annotator,ground_truth_category
0,50,2016-12-24T13:54:06.000Z,Komisi IX Ketua Komisi IX DPR RI merekomendasi...,komisi IX ketua komisi IX DPR RI rekomendasi k...,0.8,0.20,0.447214,positive
1,84,2016-12-21T08:47:51.000Z,Dapilku Rumahku Reses dan silaturahim bersama ...,dapilku rumah reses dan silaturahim sama anggo...,0.8,0.20,0.447214,positive
2,89,2016-12-20T18:32:07.000Z,mengharukan wakil rakyat trlalu banyak waktuny...,haru wakil rakyat terlalu banyak waktu untuk u...,-2.8,-0.70,0.447214,negative
3,101,2016-12-20T04:39:14.000Z,Ketua dan Para Wakil Ketua DPR RI PERLUNYA RUU...,ketua dan para wakil ketua DPR RI PERLU RUU LA...,0.8,0.20,0.447214,positive
4,121,2016-12-18T04:07:09.000Z,Viva Anggota DPR Minta TNI AU Investigasi Kece...,viva anggota DPR minta TNI AU investigasi cela...,0.6,0.15,0.547723,positive


In [4]:
# 2.3 Inisialisasi Translator Google (ID -> EN) + Preprocessing Konteks
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source='id', target='en')

# Kamus kontekstual untuk mempertahankan nuansa sentimen politik/sosial
# Gabungan antara kamus awal + kata-kata prioritas berdasarkan analisis error
KAMUS_KONTEKS = {
    # === NEGATIF: Ketidakhadiran/Kemalasan ===
    'bolos'             : 'truant absent negligent skip duty irresponsible lazy',
    'mangkir'           : 'absent irresponsible neglecting duty',
    'tak hadir'         : 'absent failed to attend irresponsible',
    'tidak hadir'       : 'absent failed to attend irresponsible',
    'sepi rapat'        : 'poorly attended negligent empty session',
    'free hatin'        : 'empty idle unproductive chamber',
    'sepi'              : 'empty deserted absent',

    # === NEGATIF: Kritik Kebijakan & Ketidakpuasan ===
    'belum tepat'        : 'inappropriate flawed not suitable misguided',
    'tidak tepat'        : 'wrong inappropriate unsuitable flawed',
    'belum selesai'      : 'unfinished delayed stalled incomplete', 
    'perlu dievaluasi'  : 'needs review problematic inadequate',
    'masih kurang'      : 'insufficient inadequate lacking',
    'molor'             : 'overdue delayed irresponsible',
    'siasat'            : 'manipulative scheme trick conspiracy',
    'miris'             : 'sad disappointing pathetic tragic',
    'konyol'            : 'ridiculous absurd foolish shameful',
    'wah masa'          : 'ridiculous absurd nonsense sarcastic',
    'ulur ulur'         : 'delaying stalling procrastinating dragging',
    'diabaikan'          : 'ignored neglected disregarded dismissed',
    'perpanjangan waktu' : 'delay postponement stalled prolonged debate',
    'mengulur ulur'      : 'stalling delaying procrastinating dragging feet',
    'perpanjangan': 'delay postponement stalled',
    
    # === NEGATIF: Pelanggaran & Kontroversi ===
    'disalahgunakan'    : 'abused misused corrupt violation',
    'penyalahgunaan'    : 'abuse misuse corruption',
    'tersangka'         : 'suspect criminal charged',
    'penipuan'          : 'fraud deception corrupt',
    'mundur'            : 'resigned controversy scandal',
    'politikkan'        : 'politicized manipulated biased',
    'elak'              : 'evading dodging avoiding accountability',
    'jangan percaya'    : 'dont trust distrust warning liar deceiver fraud',
    'bukan wakil rakyat': 'not representative betray fail corrupt selfish',
    
    # === NEGATIF: Frustrasi, Pertanyaan Retoris & Penundaan ===
    'mohon info kemajuan': 'demanding progress stalled delayed',
    'mohon info bagaimana': 'demanding info delayed stalled',
    'apakah ada'         : 'questioning doubt skeptical whether really',
    'sampai kapan'      : 'demanding accountability overdue',
    'kapan selesai'     : 'frustrated overdue stalled',
    'harus tunggu'      : 'forced wait delay frustration',
    'kaget'             : 'shocked disappointed dismayed',
    'tunggu'            : 'waiting delay stalling',
    'kapan'             : 'impatient delayed stalled',
    'wakhh'             : 'ridiculous pathetic',
    'masih belum juga'  : 'delayed stuck stagnant',
    'hoax'              : 'lying fake news fraud',
    'sampah'            : 'trash garbage waste',
    'kecewa'            : 'disappointed letdown',
    'muak'              : 'sick of tired of',
    'tipu'              : 'cheat trick fraud',
    'bohong'            : 'lie liar fake',
    'gagal'             : 'fail failure',
    'batal'             : 'cancel cancelled',
    'tunda'             : 'delay postponed',
    
    # === NEGATIF: Manipulasi & Ego ===
    'main kata'         : 'manipulative deceptive trickery',
    'di bawah tekan'    : 'oppressed pressured',
    'ego benar diri'    : 'selfish arrogant',
    'apakah banget'     : 'doubtful skeptical fake',

    # === POSITIF: Apresiasi & Kinerja ===
    'reses'             : 'constituency outreach service community visit',
    'silaturahim'       : 'community engagement meeting',
    'disahkan'          : 'successfully passed enacted accomplished',
    'disetujui'         : 'approved supported agreed',
    'merekomendasikan'  : 'recommended constructive proposed',
    'mengawal'          : 'overseeing monitoring accountable',
    'berprestasi'       : 'accomplished high performing excellent',
    'hak angket'        : 'parliamentary oversight accountability inquiry',
    
    # === POSITIF: Dukungan & Pujian (TAMBAHAN) ===
    'dukung'            : 'support endorse back strong',
    'dukung penuh'      : 'full support strong endorse',
    'semangat'          : 'spirit passion energy positive',
    'mantap'            : 'solid great awesome excellent',
    'bagus'             : 'good great fine',
    'baik'              : 'good positive favorable',
    'terima kasih'      : 'thank you grateful appreciate',
    'apresiasi'         : 'appreciation commend praise',
    'sukses'            : 'success successful accomplish',
    'hebat'             : 'great awesome amazing',
    'progres'           : 'progress advance improve',
    'tuntaskan'         : 'finish complete resolve',
    'berjalan lancar'   : 'running smooth proceeding well',
    'puji'              : 'praise compliment commend',
    'kerja bagus'       : 'good job well done',
    
    # === NETRAL: Informasi & Agenda (TAMBAHAN) ===
    # Membantu VADER mengenali konteks berita/faktual yang bukan sentimen
    'berita'            : 'news report information',
    'agenda'            : 'agenda schedule plan',
    'informasi'         : 'information info data',
    'laporan'           : 'report statement account',
    'keputusan'         : 'decision ruling verdict',
    'sidang'            : 'session hearing meeting',
    'rapat'             : 'meeting gathering session',
    'wakil rakyat'       : 'representatives lawmakers officials'
}

def preprocess_with_dictionary(text, kamus):
    """
    Mengganti kata kontekstual dengan padanan yang lebih ekspresif dalam bahasa Inggris 
    SEBELUM proses translasi dilakukan.
    """
    # lowercase agar matching dengan kunci kamus lebih mudah
    text = text.lower() 
    
    for kata_asli, kata_pengganti in kamus.items():
        # Ganti kata dengan spasi di kiri kanan agar tidak menimpa kata lain
        text = text.replace(kata_asli, f" {kata_pengganti} ")
    return text

def translate_tweet(text):
    """
    Menerjemahkan teks dengan preprocessing konteks dan retry mechanism.
    """
    if not isinstance(text, str) or pd.isna(text) or len(text.strip()) == 0:
        return ""
    
    # 1. TERAPKAN KAMUS SEBELUM TRANSLATE
    text = preprocess_with_dictionary(text, KAMUS_KONTEKS)
    
    translated = None
    retries = 0
    max_retries = 3
    
    # 2. Proses translasi dengan retry tanpa time.sleep panjang
    while translated is None and retries < max_retries:
        try:
            translated = translator.translate(text[:4500])
        except Exception as e:
            retries += 1                                                                                                                                                                                                                                  
            print(f"[RETRY {retries}] Gagal: {text[:30]}...")
    
    return translated if translated else ""

In [5]:
# 2.4 Translasi dengan Progress Bar dan Proteksi Rate Limit
print("\n[PROSES] Menerjemahkan teks ke Bahasa Inggris...")
print(f"Total data: {len(df)} tweet")
print("Estimasi waktu: ~20-40 menit(tergantung limit API & stabilitas internet)")

translated_texts = []

# Menggunakan enumerate untuk melacak indeks data (i)
for i, text in enumerate(tqdm(df['teks_processed'], desc="Translating")):
    # Panggil fungsi translasi
    translated_texts.append(translate_tweet(text))
    
    # 1. Proteksi Rate Limit: Jeda 1 detik setiap 50 tweet
    if (i + 1) % 50 == 0:
        time.sleep(1)                           
    
    # 2. Checkpoint: Simpan hasil sementara setiap 2000 tweet
    if (i + 1) % 2000 == 0:
        temp_df = df.iloc[:len(translated_texts)].copy()
        temp_df['teks_translated'] = translated_texts
        CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, f'checkpoint_{i+1}.csv')
        temp_df.to_csv(CHECKPOINT_FILE, index=False, encoding='utf-8')
        print(f"\n[CHECKPOINT] Berhasil menyimpan {i+1} data ke {CHECKPOINT_FILE}")
                                        
# Masukkan hasil akhir ke dataframe utama
df['teks_translated'] = translated_texts

print("\n[INFO] Seluruh proses translasi selesai.")


[PROSES] Menerjemahkan teks ke Bahasa Inggris...
Total data: 450 tweet
Estimasi waktu: ~20-40 menit(tergantung limit API & stabilitas internet)


Translating: 100%|██████████| 450/450 [04:13<00:00,  1.77it/s]


[INFO] Seluruh proses translasi selesai.


In [6]:
# 2.5 Validasi Hasil Translasi
null_translated = df['teks_translated'].isna().sum()
empty_translated = (df['teks_translated'].str.strip() == '').sum()

print("\n[STATISTIK] Hasil Translasi:")
print(f"  - Total tweet          : {len(df)}")
print(f"  - Gagal/Null           : {null_translated}")
print(f"  - Kosong setelah translasi: {empty_translated}")
print(f"  - Berhasil diterjemahkan: {len(df) - null_translated - empty_translated}")

# Preview hasil
print("\n[PREVIEW] 3 Tweet Pertama:")
for i in range(3):
    print(f"\n🇮🇩 Processed : {df['teks_processed'].iloc[i][:80]}...")
    print(f"🇬🇧 Translated: {df['teks_translated'].iloc[i][:80]}...")


[STATISTIK] Hasil Translasi:
  - Total tweet          : 450
  - Gagal/Null           : 0
  - Kosong setelah translasi: 0
  - Berhasil diterjemahkan: 450

[PREVIEW] 3 Tweet Pertama:

🇮🇩 Processed : komisi IX ketua komisi IX DPR RI rekomendasi kepada pem agar tinjau ulang kebija...
🇬🇧 Translated: Commission IX Chairman of Commission IX DPR RI recommends to the government to r...

🇮🇩 Processed : dapilku rumah reses dan silaturahim sama anggota DPR RI pkbmembelarakyat...
🇬🇧 Translated: my electoral district, home, constituency, outreach service, community visit and...

🇮🇩 Processed : haru wakil rakyat terlalu banyak waktu untuk update disosmed hampir tidak ada wa...
🇬🇧 Translated: haru representatives legislators have too much time to update on social media, t...


In [7]:
# Mencari baris dengan hasil translasi kosong
empty_mask = df['teks_translated'].isna() | (df['teks_translated'].str.strip() == '')
df_kosong = df[empty_mask]

print(f" Ditemukan {len(df_kosong)} baris kosong:\n")
for _, row in df_kosong.iterrows():
    print(f"No   : {row['no']}")
    print(f"Asli : {row['teks_processed'][:120]}...")
    print(f"Hasil: '{row['teks_translated']}'")
    print("-" * 60)

 Ditemukan 0 baris kosong:



In [8]:
# 2.6 Simpan Data Hasil Translasi
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'data_translated.csv')
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[OUTPUT] Data berhasil disimpan ke: {OUTPUT_FILE}")


[OUTPUT] Data berhasil disimpan ke: ../../outputs/VTB\data_translated.csv
